In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Incremental RelShift, DSpar, and LSP Production Runner

This notebook rebuilds pruning, training, and comparison outputs from scratch in Colab. The working directory is fixed to `/content/structure_faithful_gnn/`.

Execution order is fixed: incremental RelShift, DSpar, LSP, common grid, selected training, master results, comparison figures, export.

## 1. Configuration

Set the repository, datasets, models, comparison targets, and output roots. `RESET_EXPERIMENT_OUTPUTS=True` removes the configured Colab output folders before running so the produced artifacts are fresh.

In [ ]:
from pathlib import Path

GITHUB_REPO = "https://github.com/sasipilav/BBM462-Structure-Faithful-GNN-Sparsification"
GITHUB_BRANCH = "main"
GITHUB_TOKEN_SECRET_NAME = "GITHUB_TOKEN"
WORKDIR = Path("/content/structure_faithful_gnn")

DATASET_CONFIGS = [
    "configs/datasets/cora.yaml",
    "configs/datasets/citeseer.yaml",
]
MODEL_CONFIGS = [
    "configs/models/gcn.yaml",
    "configs/models/graphsage.yaml",
]
RELSHIFT_PRUNING_CONFIG = "configs/pruning/relshift_incremental_sequential.yaml"

COMPARISON_TARGETS = {
    "cora": [0.02, 0.05, 0.08, 0.11, 0.14, 0.17, 0.20, 0.23, 0.26, 0.29, 0.32, 0.35, 0.38, 0.41, 0.44],
    "citeseer": [0.02, 0.05, 0.08, 0.11, 0.14, 0.17, 0.20, 0.23, 0.26, 0.29, 0.32, 0.35],
}

PRUNING_SEED = 0
TRAINING_SEEDS = [0, 1, 2]
MAIN_COMPARABILITY_GAP = 0.015
AUX_COMPARABILITY_GAP = 0.02
TRAINING_INCLUDE_STATUS = ["main_comparable"]
WRITE_RELSHIFT_EDGE_SCORES = False  # Enable only for RelShift score-analysis runs; files are large.

RESULTS_ROOT = WORKDIR / "results"
RELSHIFT_FRONTIER_ROOT = RESULTS_ROOT / "frontiers" / "incremental_relshift"
DSPAR_FRONTIER_ROOT = RESULTS_ROOT / "frontiers" / "dspar"
LSP_FRONTIER_ROOT = RESULTS_ROOT / "frontiers" / "lsp"
COMMON_GRID_ROOT = RESULTS_ROOT / "analysis" / "common_grid"
TRAINING_ROOT = RESULTS_ROOT / "frontier_training"
DENSE_REFERENCE_ROOT = RESULTS_ROOT / "frontier_dense"
MASTER_RESULTS_ROOT = RESULTS_ROOT / "analysis" / "frontier_master"
FRONTIER_ANALYSIS_ROOT = RESULTS_ROOT / "analysis" / "frontier_comparison"
RUNTIME_ANALYSIS_ROOT = RESULTS_ROOT / "analysis" / "runtime_comparison"

RESET_EXPERIMENT_OUTPUTS = True
DRIVE_EXPORT_DIR = Path("/content/drive/MyDrive/structure_faithful_gnn_outputs")

## 2. Helpers

These helpers run shell commands, resolve an optional GitHub token from Colab secrets, clone or update the repo, and format dataset-specific target flags.

In [ ]:
import os
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

try:
    from google.colab import userdata
except Exception:
    userdata = None

def q(value):
    return shlex.quote(str(value))

def run(cmd, cwd=None, env=None):
    print(f"+ {cmd}")
    completed = subprocess.run(
        cmd,
        shell=True,
        cwd=cwd,
        env=env,
        text=True,
        capture_output=True,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.returncode != 0:
        if completed.stderr:
            print(completed.stderr)
        raise subprocess.CalledProcessError(completed.returncode, completed.args)
    if completed.stderr:
        print(completed.stderr)
    return completed

def dataset_target_flags(targets):
    parts = []
    for dataset, values in targets.items():
        joined = ",".join(str(value) for value in values)
        parts.append(f"--comparison-target-set {q(f'{dataset}:{joined}')}")
    return " ".join(parts)

def resolve_optional_github_token(secret_name):
    if userdata is None:
        return None
    try:
        return userdata.get(secret_name)
    except Exception:
        return None

def git_env_with_optional_token(token):
    env = os.environ.copy()
    if not token:
        return env
    askpass = Path("/tmp/git_askpass.sh")
    askpass.write_text('#!/bin/sh\necho "$GITHUB_TOKEN"\n', encoding="utf-8")
    askpass.chmod(0o700)
    env["GIT_ASKPASS"] = str(askpass)
    env["GITHUB_TOKEN"] = token
    return env

def prepare_repo(repo_url, branch, dest):
    token = resolve_optional_github_token(GITHUB_TOKEN_SECRET_NAME)
    env = git_env_with_optional_token(token)
    if dest.exists() and (dest / ".git").exists():
        run(f"git -C {q(dest)} fetch origin {q(branch)}", env=env)
        run(f"git -C {q(dest)} checkout {q(branch)}", env=env)
        run(f"git -C {q(dest)} pull --ff-only origin {q(branch)}", env=env)
        return
    if dest.exists():
        raise RuntimeError(f"{dest} exists but is not a git checkout. Remove it or choose another WORKDIR.")
    run(f"git clone --depth 1 --branch {q(branch)} {q(repo_url)} {q(dest)}", env=env)

## 3. Clone Or Update Repository

In [ ]:
prepare_repo(GITHUB_REPO, GITHUB_BRANCH, WORKDIR)
print({"WORKDIR": str(WORKDIR)})

## 4. Install Dependencies And Build Native Components

This installs project dependencies, the PyG companion wheels required by weighted GraphSAGE runs, ORCA for GDV counting, and the native exact incremental RelShift extension.

In [ ]:
%%capture

run(f"{sys.executable} -m pip install -r requirements.txt", cwd=WORKDIR)

import importlib
import torch

torch_version = torch.__version__.split("+")[0]
cuda_tag = f"cu{torch.version.cuda.replace('.', '')}" if torch.version.cuda else "cpu"
pyg_wheel = f"https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html"
run(
    f"{sys.executable} -m pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f {q(pyg_wheel)}",
    cwd=WORKDIR,
)
for module_name in ("torch_scatter", "torch_sparse"):
    importlib.import_module(module_name)
print({"pyg_wheel": pyg_wheel, "verified_modules": ["torch_scatter", "torch_sparse"]})

orca_dir = WORKDIR / "tmp" / "orca"
if not orca_dir.exists():
    run(f"git clone --depth 1 https://github.com/thocevar/orca {q(orca_dir)}")
orca_cpp = orca_dir / "orca.cpp"
orca_bin = orca_dir / "orca"
if not orca_cpp.exists():
    raise FileNotFoundError(f"Expected ORCA source file at {orca_cpp}")
run(f"g++ -O2 -std=c++11 -o {q(orca_bin)} {q(orca_cpp)}", cwd=orca_dir)
os.environ["ORCA_BINARY"] = str(orca_bin)
print({"ORCA_BINARY": os.environ["ORCA_BINARY"]})

run(f"{sys.executable} scripts/build_relshift_incremental_extension.py --smoke", cwd=WORKDIR)

## 5. Reset Output Roots

In [ ]:
if RESET_EXPERIMENT_OUTPUTS and RESULTS_ROOT.exists():
    shutil.rmtree(RESULTS_ROOT)
    print({"removed": str(RESULTS_ROOT)})
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

## 6. Prune: Incremental RelShift

In [ ]:
cmd = (
    f"{sys.executable} scripts/run_relshift_calibration.py "
    f"--datasets {' '.join(q(path) for path in DATASET_CONFIGS)} "
    f"--pruning {q(RELSHIFT_PRUNING_CONFIG)} "
    f"--seed {PRUNING_SEED} "
    f"{dataset_target_flags(COMPARISON_TARGETS)} "
    f"--output-root {q(RELSHIFT_FRONTIER_ROOT)}"
    f"{' --write-edge-scores' if WRITE_RELSHIFT_EDGE_SCORES else ''}"
)
run(cmd, cwd=WORKDIR)

## 7. Prune: DSpar

In [ ]:
cmd = (
    f"{sys.executable} scripts/run_dspar_calibration.py "
    f"--datasets {' '.join(q(path) for path in DATASET_CONFIGS)} "
    f"--seed {PRUNING_SEED} "
    f"{dataset_target_flags(COMPARISON_TARGETS)} "
    f"--output-root {q(DSPAR_FRONTIER_ROOT)}"
)
run(cmd, cwd=WORKDIR)

## 8. Prune: LSP

In [ ]:
cmd = (
    f"{sys.executable} scripts/run_lsp_calibration.py "
    f"--datasets {' '.join(q(path) for path in DATASET_CONFIGS)} "
    f"--seed {PRUNING_SEED} "
    f"{dataset_target_flags(COMPARISON_TARGETS)} "
    f"--output-root {q(LSP_FRONTIER_ROOT)}"
)
run(cmd, cwd=WORKDIR)

## 9. Build Common Attainable Grid

In [ ]:
cmd = (
    f"{sys.executable} scripts/build_common_attainable_grid.py "
    f"--datasets {' '.join(q(path) for path in DATASET_CONFIGS)} "
    f"--relshift-frontier {q(RELSHIFT_FRONTIER_ROOT / 'frontier.csv')} "
    f"--dspar-frontier {q(DSPAR_FRONTIER_ROOT / 'frontier.csv')} "
    f"--lsp-frontier {q(LSP_FRONTIER_ROOT / 'frontier.csv')} "
    f"--main-gap {MAIN_COMPARABILITY_GAP} --aux-gap {AUX_COMPARABILITY_GAP} "
    f"{dataset_target_flags(COMPARISON_TARGETS)} "
    f"--output-root {q(COMMON_GRID_ROOT)}"
)
run(cmd, cwd=WORKDIR)

## 10. Train Selected Comparable Points

In [ ]:
cmd = (
    f"{sys.executable} scripts/run_training_manifest.py "
    f"--manifest {q(COMMON_GRID_ROOT / 'training_manifest.csv')} "
    f"--models {' '.join(q(path) for path in MODEL_CONFIGS)} "
    f"--seeds {' '.join(str(seed) for seed in TRAINING_SEEDS)} "
    f"--include-status {' '.join(q(status) for status in TRAINING_INCLUDE_STATUS)} "
    f"--output-root {q(TRAINING_ROOT)} "
    f"--dense-output-root {q(DENSE_REFERENCE_ROOT)}"
)
run(cmd, cwd=WORKDIR)

## 11. Build Master Results

In [ ]:
cmd = (
    f"{sys.executable} scripts/build_master_results_table.py "
    f"--relshift-root {q(TRAINING_ROOT)} "
    f"--dspar-root {q(TRAINING_ROOT)} "
    f"--lsp-root {q(TRAINING_ROOT)} "
    f"--output-root {q(MASTER_RESULTS_ROOT)}"
)
run(cmd, cwd=WORKDIR)

## 12. Build Comparison Tables And Figures

In [ ]:
cmd = (
    f"{sys.executable} scripts/analyze_frontier_comparison.py "
    f"--master-results {q(MASTER_RESULTS_ROOT / 'master_results.csv')} "
    f"--common-grid {q(COMMON_GRID_ROOT / 'common_attainable_grid.csv')} "
    f"--output-root {q(FRONTIER_ANALYSIS_ROOT)}"
)
run(cmd, cwd=WORKDIR)

## 13. Build Runtime Tables

In [ ]:
cmd = (
    f"{sys.executable} scripts/analyze_runtime_comparison.py "
    f"--master-results {q(MASTER_RESULTS_ROOT / 'master_results.csv')} "
    f"--common-grid {q(COMMON_GRID_ROOT / 'common_attainable_grid.csv')} "
    f"--relshift-frontier {q(RELSHIFT_FRONTIER_ROOT / 'frontier.csv')} "
    f"--dspar-frontier {q(DSPAR_FRONTIER_ROOT / 'frontier.csv')} "
    f"--lsp-frontier {q(LSP_FRONTIER_ROOT / 'frontier.csv')} "
    f"--output-root {q(RUNTIME_ANALYSIS_ROOT)}"
)
run(cmd, cwd=WORKDIR)

## 14. Preview Key Outputs

In [ ]:
import pandas as pd

preview_paths = [
    RELSHIFT_FRONTIER_ROOT / "frontier.csv",
    DSPAR_FRONTIER_ROOT / "frontier.csv",
    LSP_FRONTIER_ROOT / "frontier.csv",
    COMMON_GRID_ROOT / "common_attainable_grid.csv",
    COMMON_GRID_ROOT / "method_coverage_summary.csv",
    TRAINING_ROOT / "training_manifest_summary.csv",
    MASTER_RESULTS_ROOT / "master_results.csv",
    FRONTIER_ANALYSIS_ROOT / "matched_three_way_comparison.csv",
    FRONTIER_ANALYSIS_ROOT / "pairwise_summary.csv",
    RUNTIME_ANALYSIS_ROOT / "frontier_pruning_runtime_summary.csv",
    RUNTIME_ANALYSIS_ROOT / "matched_pruning_runtime_summary.csv",
    RUNTIME_ANALYSIS_ROOT / "training_runtime_summary.csv",
    RUNTIME_ANALYSIS_ROOT / "end_to_end_runtime_summary.csv",
    RUNTIME_ANALYSIS_ROOT / "runtime_pairwise_summary.csv",
]

for path in preview_paths:
    print("\n", path)
    if path.exists():
        display(pd.read_csv(path).head())
    else:
        print({"missing": str(path)})

## 15. Zip Results And Copy To Drive

In [ ]:
import datetime as dt
import shutil

timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
zip_base = Path("/content") / f"structure_faithful_gnn_results_{timestamp}"
archive_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=WORKDIR, base_dir="results"))
DRIVE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
drive_path = DRIVE_EXPORT_DIR / archive_path.name
shutil.copy2(archive_path, drive_path)
print({"zip_path": str(archive_path), "drive_copy": str(drive_path)})